<a href="https://colab.research.google.com/github/geopayme/AstroPhysics/blob/main/Combined_PDF_Signer_and_Verifier_OpenSSL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🔏 PDF Signing & Authenticity Validator with Metadata Extraction
This notebook allows you to:
- Upload a preprint PDF
- Extract metadata (email, ORCID, institution)
- Generate full SHA-256 hash and embed it in:
  - The last page (as a visible stamp)
  - A `.sha256` file for archive
  - PDF metadata (custom field)
- Re-validate authenticity by comparing hashes

In [ ]:
# 📦 Install PyMuPDF
!pip install PyMuPDF

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.0/20.0 MB 31.9 MB/s eta 0:00:00


In [ ]:
# 📤 Upload PDF
from google.colab import files
uploaded = files.upload()
pdf_path = list(uploaded.keys())[0]
print(f'Uploaded: {pdf_path}')

In [ ]:
# 🔎 Extract metadata from first page
import fitz, re
doc = fitz.open(pdf_path)
first_page_text = doc[0].get_text()
email = re.findall(r'[\w\.-]+@[\w\.-]+\.\w+', first_page_text)
orcid = re.findall(r'\d{4}-\d{4}-\d{4}-\d{4}', first_page_text)
inst = re.findall(r'.*(University|Institute|Department|College).*', first_page_text, re.IGNORECASE)
email_guess = email[0] if email else ''
orcid_guess = orcid[0] if orcid else ''
inst_guess = inst[0] if inst else ''
doc.close()
print('Email:', email_guess)
print('ORCID:', orcid_guess)
print('Institution:', inst_guess)

In [ ]:
# ✏️ Author Data Input
from datetime import datetime
author = input('Author Name: ')
email = input(f'Email [{email_guess}]: ') or email_guess
orcid = input(f'ORCID [{orcid_guess}]: ') or orcid_guess
institution = input(f'Institution [{inst_guess}]: ') or inst_guess
timestamp = datetime.utcnow().isoformat() + 'Z'

In [ ]:
# 🔐 Generate SHA-256 Hash
import hashlib
with open(pdf_path, 'rb') as f:
    pdf_bytes = f.read()
    sha256_hash = hashlib.sha256(pdf_bytes).hexdigest()
print('✅ SHA-256:', sha256_hash)

In [ ]:
# 🖋️ Stamp Last Page + Auto-Clean Old Footers
doc = fitz.open(pdf_path)
last = doc[-1]

# 🧹 Clean previous signature blocks
blocks_to_remove = ["Signed by:", "SHA-256:", "ORCID:", email]
for block in last.get_text("dict")["blocks"]:
    for line in block.get("lines", []):
        for span in line.get("spans", []):
            if any(keyword in span["text"] for keyword in blocks_to_remove):
                rect = fitz.Rect(span["bbox"])
                last.add_redact_annot(rect, fill=(1, 1, 1))
last.apply_redactions()

# 🖋️ Insert new clean footer
stamp = f"""Signed by: {author}\nEmail: {email}\nORCID: {orcid}\nInstitution: {institution}\nDate: {timestamp}\nSHA-256: {sha256_hash}"""
last.insert_textbox(fitz.Rect(50, last.rect.height - 130, 550, last.rect.height - 20), stamp, fontsize=8, fontname="helv", color=(0, 0, 0))
doc.set_metadata({
  "title": "Signed Preprint",
  "subject": f"SHA-256:{sha256_hash}",
  "keywords": f"signature, integrity, sha256:{sha256_hash}",
  "author": author
})
signed_file = 'signed_' + pdf_path
doc.save(signed_file)
doc.close()
print(f'Saved: {signed_file}')

In [ ]:
# 📁 Save .sha256 file
sha_file = signed_file + '.sha256'
with open(sha_file, 'w') as f:
    f.write(f"{sha256_hash}  {signed_file}\n")
print(f'SHA256 saved: {sha_file}')

In [ ]:
# ⬇️ Download signed PDF + hash
files.download(signed_file)
files.download(sha_file)

## 🔍 Validate PDF Authenticity

In [ ]:
# 🆚 Upload PDF to validate
uploaded = files.upload()
verify_pdf = list(uploaded.keys())[0]

In [ ]:
# 🧾 Upload hash or paste manually
try:
  uploaded = files.upload()
  sha_file = list(uploaded.keys())[0]
  with open(sha_file, 'r') as f:
    expected_hash = f.read().strip().split()[0]
except:
  expected_hash = input('Paste expected SHA-256 hash: ')

In [ ]:
# ✅ Validate SHA-256 hash
with open(verify_pdf, 'rb') as f:
    file_hash = hashlib.sha256(f.read()).hexdigest()
print('Expected:', expected_hash)
print('Computed:', file_hash)
if file_hash == expected_hash:
    print('✅ Authentic: The file has not been altered.')
else:
    print('❌ Not authentic: Hash mismatch.')

# 🔐 Cryptographic Signing (RSA Signature of PDF Hash)
Upload your private RSA `.pem` key and the generated `.sha256` hash to sign the hash cryptographically.

In [ ]:
# 📤 Upload private RSA key (.pem) and .sha256 file
uploaded = files.upload()
sha256_file = [f for f in uploaded if f.endswith('.sha256')][0]
private_key_file = [f for f in uploaded if f.endswith('.pem')][0]

In [ ]:
# 🖋️ Sign the SHA-256 file using private key
signed_hash_file = sha256_file + '.sig'
!openssl dgst -sha256 -sign {private_key_file} -out {signed_hash_file} {sha256_file}
print(f'Signature saved to: {signed_hash_file}')

In [ ]:
# 📥 Download signature file
files.download(signed_hash_file)

# ✅ Verify Signature Against PDF Hash
Use your public key to confirm that the `.sig` matches the `.sha256` hash.

In [ ]:
# 📤 Upload .sig file and public RSA key (.pem)
uploaded = files.upload()
sig_file = [f for f in uploaded if f.endswith('.sig')][0]
public_key_file = [f for f in uploaded if f.endswith('.pem')][0]

In [ ]:
# ✅ Verify that the signature matches the hash
cmd = f"openssl dgst -sha256 -verify {public_key_file} -signature {sig_file} {sha256_file}"
print("Running:", cmd)
!$cmd